# 📖 Notebook 2: Conflict Resolution with CRDTs

In the previous notebook we learned about OT, where a **central server** transforms operations to keep everyone in sync. CRDTs (Conflict-free Replicated Data Types) take a completely different approach: they make operations **commutative** — meaning they can be applied in **any order** and still produce the same result.

## Learning Objectives

By the end of this notebook, you'll understand:
- What CRDTs are and why they exist
- How position-based IDs make inserts order-independent
- How tombstones handle deletions
- OT vs CRDT trade-offs for real systems

## 🛠️ Setup

```bash
cd 06-system-designs/google-docs
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

## 🤔 Why CRDTs?

OT requires a **central server** — every operation must pass through it so the server can enforce ordering. But what if:

- You want **offline editing** (like Apple Notes)?
- You want **peer-to-peer** collaboration (no server)?
- You want to **scale beyond one server** per document?

CRDTs solve this by guaranteeing that **no matter what order operations arrive, every client converges on the same document**.

```
OT:    "Apply operations in the right order"     → needs a server
CRDTs: "Make operations work in ANY order"        → no server needed
```

## 🔑 The Three Key Tricks of Text CRDTs

### Trick 1: Unique Position IDs

Instead of using integer positions (like `INSERT(5, "x")`), CRDTs give every character a **unique ID** that can be infinitely subdivided — like decimal numbers between 0 and 1.

```
Traditional (OT):     H  e  l  l  o  !
                      0  1  2  3  4  5    ← integer positions (shift when you insert!)

CRDT:                 H    e    l    l    o    !
                     0.1  0.2  0.3  0.4  0.5  0.6   ← unique IDs (never change!)
```

To insert between `o` (0.5) and `!` (0.6), just pick a number between them: 0.55.

### Trick 2: Tombstones for Deletion

When you delete a character, you don't actually remove it. Instead, you **mark it as deleted** (a "tombstone"). The character is hidden from view but stays in the data structure to maintain position references.

```
Before delete:  H  e  l  l  o  !     (all visible)
After delete !: H  e  l  l  o  [!]   (! is tombstoned, hidden from display)
```

### Trick 3: A Stable Identity Per Character

"Pick a number between them" is only half a design. Two sites editing offline can
pick the **same** number, and any real network will deliver the same message
twice. So every character also carries a `(site_id, counter)` **char id**, which is:

- the tie-break when two position ids collide — and it has to be the *character's*
  id, not the id of whichever replica happens to be doing the sorting, or the two
  replicas sort the same characters differently;
- the deduplication key — re-applying an operation you have already seen must be
  a no-op, or a retried message duplicates a character;
- what a delete names. Deletes say "tombstone character `(bob, 7)`", never
  "tombstone whatever is at 0.55".

The next cell shows why the position id also has to be an **exact** number rather
than a float.

In [ ]:
# ⚠️ Why the position ids cannot be floats
#
# "Pick a number between the neighbours" only works if you can keep subdividing.
# Typing at the end of a document does exactly that, over and over, and a float
# interval runs out of room surprisingly fast.

before = 0.0
collapsed_at = None
for i in range(1, 200):
    nxt = before + (1.0 - before) * 0.5      # halfway to the end of the document
    if nxt == before:                        # no float left between them
        collapsed_at = i
        break
    before = nxt

print(f"float: after {collapsed_at} inserts at the end, the interval collapsed.")
print(f"       character {collapsed_at} would get id {before!r} — the same id as character {collapsed_at - 1}.")
print("       Two characters with one id means a delete can tombstone the wrong one.")
print()

from fractions import Fraction

exact = Fraction(0)
for i in range(1, 200):
    exact = exact + (Fraction(1) - exact) * Fraction(1, 2)

print(f"Fraction: after 199 inserts the id is still strictly inside the interval.")
print(f"          numerator/denominator are now {len(str(exact.denominator))}-digit numbers,")
print(f"          and there is still room for infinitely many more characters.")

assert collapsed_at is not None and collapsed_at < 100, (
    "floats stopped collapsing — the motivation for exact rationals no longer holds"
)
assert exact < 1, "exact rational arithmetic left the interval — impossible"
print()
print("💡 So the CRDT below stores position ids as exact rationals, not floats.")

In [ ]:
import random
from fractions import Fraction


class CRDTChar:
    """A single character in our CRDT document.

    Each character has:
    - position_id: an exact rational that fixes its place in the document
    - char_id: (site_id, counter) — globally unique, and the same on every replica
    - char: the actual character
    - deleted: whether this character has been tombstoned
    """
    def __init__(self, position_id, char, char_id, deleted=False):
        self.position_id = position_id
        self.char = char
        self.char_id = char_id
        self.deleted = deleted

    @property
    def site_id(self):
        return self.char_id[0]

    def sort_key(self):
        """Total order over characters, identical on every replica.

        Two sites can independently pick the same position_id, so position
        alone is not a total order. The char_id breaks the tie — and note it
        is the *character's* id, never the local site's: sorting must not
        depend on who is doing the sorting.
        """
        return (self.position_id, self.char_id)

    def __repr__(self):
        status = "🪦" if self.deleted else "✓"
        return f"{status} '{self.char}' @{float(self.position_id):.4f} (site={self.site_id})"


class CRDTDocument:
    """A simple CRDT-based collaborative text document.

    This is a simplified version of algorithms like LSEQ or Logoot. Real
    implementations use variable-length ids rather than rationals, but the
    properties we care about are the same:

      commutative  — a then b gives the same document as b then a
      associative  — so any delivery order at all converges
      idempotent   — delivering the same operation twice changes nothing
    """

    def __init__(self, site_id):
        self.site_id = site_id
        self.chars = []       # kept sorted by sort_key()
        self.counter = 0      # per-site sequence number, makes char_ids unique
        self.applied = set()  # char_ids already inserted → idempotent inserts
        self.tombstones = set()  # char_ids deleted, even if not yet inserted here

    def _insert_char(self, crdt_char):
        """Insert into the sorted list. Returns False if this char is already here."""
        if crdt_char.char_id in self.applied:
            return False      # duplicate delivery — a CRDT must ignore it
        self.applied.add(crdt_char.char_id)
        if crdt_char.char_id in self.tombstones:
            # its delete arrived before it did; stay deleted
            crdt_char.deleted = True
        key = crdt_char.sort_key()
        i = 0
        while i < len(self.chars) and self.chars[i].sort_key() < key:
            i += 1
        self.chars.insert(i, crdt_char)
        return True

    def _visible(self):
        return [c for c in self.chars if not c.deleted]

    def _generate_position_id(self, index):
        """Pick an id strictly between the visible neighbours of `index`.

        Exact rationals, so the gap can always be subdivided again. Two sites
        editing the same gap can still land on the same id — that is what the
        char_id tie-break in sort_key() is for.
        """
        visible = self._visible()
        before = visible[index - 1].position_id if index > 0 else Fraction(0)
        after = visible[index].position_id if index < len(visible) else Fraction(1)
        return before + (after - before) * Fraction(random.randint(3, 7), 10)

    def insert(self, index, char):
        """Insert a character at visible index. Returns the operation."""
        self.counter += 1
        char_id = (self.site_id, self.counter)
        pos_id = self._generate_position_id(index)
        self._insert_char(CRDTChar(pos_id, char, char_id))
        return {"type": "insert", "position_id": pos_id, "char": char, "char_id": char_id}

    def delete(self, index):
        """Delete the character at visible index. Returns the operation."""
        visible = self._visible()
        if index >= len(visible):
            return None
        target = visible[index]
        target.deleted = True
        self.tombstones.add(target.char_id)
        # The op names the CHARACTER, not a position. Positions can collide;
        # char_ids cannot, so every replica tombstones exactly the same char.
        return {"type": "delete", "char_id": target.char_id, "site_id": self.site_id}

    def apply_remote(self, op):
        """Apply a remote operation. Safe in any order, and safe to apply twice."""
        if op["type"] == "insert":
            self._insert_char(CRDTChar(op["position_id"], op["char"], op["char_id"]))
        elif op["type"] == "delete":
            self.tombstones.add(op["char_id"])
            for c in self.chars:
                if c.char_id == op["char_id"]:
                    c.deleted = True      # already True stays True → idempotent
                    break

    def get_text(self):
        """Get the visible text (excluding tombstoned characters)."""
        return "".join(c.char for c in self.chars if not c.deleted)

    def get_all_chars(self):
        """Get all characters including tombstones (for debugging)."""
        return self.chars


print("CRDT Document class defined! ✅")

In [ ]:
# Demo: Basic CRDT operations
random.seed(42)  # for reproducible positions

doc = CRDTDocument(site_id="alice")

# Build "Hello!" one character at a time
for i, ch in enumerate("Hello!"):
    doc.insert(i, ch)

print(f"Document text: '{doc.get_text()}'")
print(f"\nInternal representation (with position IDs):")
for c in doc.get_all_chars():
    print(f"  {c}")

print(f"\n💡 Notice each character has a unique position_id.")
print(f"   These IDs NEVER change, even when other chars are inserted.")

assert doc.get_text() == "Hello!", doc.get_text()
position_ids = [c.position_id for c in doc.get_all_chars()]
assert len(set(position_ids)) == len(position_ids), "position ids collided — they must be unique"
assert position_ids == sorted(position_ids), "the char list is not sorted by position id"

In [ ]:
# Demo: Tombstones — deleting keeps the character but marks it invisible
random.seed(42)

doc = CRDTDocument(site_id="alice")
for i, ch in enumerate("Hello!"):
    doc.insert(i, ch)

print(f"Before delete: '{doc.get_text()}'")
print(f"Chars in memory: {len(doc.get_all_chars())}")
print()

# Delete the '!' (visible index 5)
op = doc.delete(5)
print(f"After deleting '!': '{doc.get_text()}'")
print(f"Chars in memory: {len(doc.get_all_chars())}  ← still 6!")
print()

print("Internal state:")
for c in doc.get_all_chars():
    print(f"  {c}")

print(f"\n💡 The '!' is still in memory (tombstoned 🪦) but hidden from display.")
print(f"   This is the trade-off: CRDTs use more memory because documents only grow.")

assert doc.get_text() == "Hello", doc.get_text()
assert len(doc.get_all_chars()) == 6, "the tombstone was removed instead of hidden"
# The delete op names a character, not a position — that is what makes it safe
# to replay on a replica whose position ids happen to collide.
assert op["char_id"] == doc.get_all_chars()[-1].char_id

## 🔄 Concurrent Editing with CRDTs

The magic of CRDTs: two users can edit simultaneously, receive each other's operations in **any order**, and still converge on the same document.

In [ ]:
# Two users editing the same document concurrently
random.seed(100)

# Both start with the same document
alice_doc = CRDTDocument(site_id="alice")
bob_doc = CRDTDocument(site_id="bob")

# Initialize both with "Hello"
init_ops = []
for i, ch in enumerate("Hello"):
    op = alice_doc.insert(i, ch)
    init_ops.append(op)

# Sync initial state to Bob
for op in init_ops:
    bob_doc.apply_remote(op)

print(f"Alice's document: '{alice_doc.get_text()}'")
print(f"Bob's document:   '{bob_doc.get_text()}'")
print()

# Now both edit CONCURRENTLY (without seeing each other's changes)
# Alice inserts ", world" at position 5
alice_ops = []
for i, ch in enumerate(", world"):
    op = alice_doc.insert(5 + i, ch)
    alice_ops.append(op)

# Bob inserts "!" at position 5
bob_ops = []
op = bob_doc.insert(5, "!")
bob_ops.append(op)

print(f"Alice (before sync): '{alice_doc.get_text()}'")
print(f"Bob (before sync):   '{bob_doc.get_text()}'")
print()

# Now sync: Alice receives Bob's ops, Bob receives Alice's ops
for op in bob_ops:
    alice_doc.apply_remote(op)
for op in alice_ops:
    bob_doc.apply_remote(op)

print(f"Alice (after sync):  '{alice_doc.get_text()}'")
print(f"Bob (after sync):    '{bob_doc.get_text()}'")
print()

converged = alice_doc.get_text() == bob_doc.get_text()
print(f"✅ Documents converged: {converged}")
print(f"\n💡 No central server needed! Both clients applied operations")
print(f"   independently and arrived at the same result.")
print()
print("⚠️  Look at where Bob's '!' ended up. Alice typed ', world' as seven")
print("   separate one-character inserts into the same gap, and Bob's insert")
print("   landed *inside* that run. This is the CRDT interleaving anomaly: the")
print("   merge is convergent, but it is not what either user meant. Production")
print("   CRDTs (RGA, Yjs) cut it down by referencing 'insert after character X'")
print("   instead of picking a number in a gap — they do not eliminate it.")

assert converged, f"CRDT diverged: {alice_doc.get_text()!r} vs {bob_doc.get_text()!r}"
# Convergence alone is not enough — nobody's characters may go missing either.
merged = alice_doc.get_text()
assert sorted(merged) == sorted("Hello" + ", world" + "!"), f"characters were lost or duplicated: {merged!r}"

In [ ]:
# Let's prove order doesn't matter: apply ops in DIFFERENT orders
random.seed(200)

# Create three independent documents
doc_a = CRDTDocument(site_id="alice")
doc_b = CRDTDocument(site_id="bob")
doc_c = CRDTDocument(site_id="charlie")

# Initialize all with "ABC"
init_ops = []
for i, ch in enumerate("ABC"):
    op = doc_a.insert(i, ch)
    init_ops.append(op)
for op in init_ops:
    doc_b.apply_remote(op)
    doc_c.apply_remote(op)

# Three concurrent edits:
op1 = doc_a.insert(1, "x")   # Alice inserts 'x' after 'A'
op2 = doc_b.insert(2, "y")   # Bob inserts 'y' after 'B'
op3 = doc_c.insert(3, "z")   # Charlie inserts 'z' after 'C'

# Apply in different orders to each document
# Doc A: already has op1, apply op2 then op3
doc_a.apply_remote(op2)
doc_a.apply_remote(op3)

# Doc B: already has op2, apply op3 then op1 (DIFFERENT ORDER)
doc_b.apply_remote(op3)
doc_b.apply_remote(op1)

# Doc C: already has op3, apply op1 then op2 (DIFFERENT ORDER)
doc_c.apply_remote(op1)
doc_c.apply_remote(op2)

print("Three users, three different operation orders:")
print(f"  Alice   (op1 → op2 → op3): '{doc_a.get_text()}'")
print(f"  Bob     (op2 → op3 → op1): '{doc_b.get_text()}'")
print(f"  Charlie (op3 → op1 → op2): '{doc_c.get_text()}'")
print()

all_same = doc_a.get_text() == doc_b.get_text() == doc_c.get_text()
print(f"✅ All documents identical: {all_same}")
print(f"\n💡 This is the CRDT guarantee: convergence regardless of operation order!")

assert all_same, (
    f"CRDT diverged: {doc_a.get_text()!r} / {doc_b.get_text()!r} / {doc_c.get_text()!r}"
)

## 🧪 The Three Properties, Actually Checked

"Operations commute" is a claim with a precise meaning. For a CRDT the merge has
to be:

| Property | Meaning | What breaks without it |
|----------|---------|------------------------|
| **Commutative** | `a` then `b` == `b` then `a` | two clients that got the same two edits in different orders show different text |
| **Associative** | grouping doesn't matter, so *any* interleaving converges | batching or syncing in chunks changes the result |
| **Idempotent** | applying the same op twice == applying it once | any retry — and every real network retries — duplicates a character |

Three worked examples don't establish any of those. The cell below checks all
three over **random delivery orders that include deletes**, plus duplicate and
out-of-order delivery, and asserts convergence rather than printing a hopeful
tick.

In [ ]:
# Commutativity + associativity + idempotence, over random delivery orders.
import random

def make_concurrent_ops(seed):
    """Three sites edit the same 'ABCDE' document without seeing each other."""
    random.seed(seed)
    origin = CRDTDocument(site_id="origin")
    base_ops = [origin.insert(i, ch) for i, ch in enumerate("ABCDE")]

    edits = []
    for site in ("alice", "bob", "carol"):
        replica = CRDTDocument(site_id=site)
        for op in base_ops:
            replica.apply_remote(op)
        edits.append(replica.insert(random.randrange(6), site[0].upper()))
        edits.append(replica.delete(random.randrange(5)))
    return base_ops + edits


def replay(ops, tag, duplicate=False):
    """Feed ops to a fresh replica in the given order; optionally twice each."""
    replica = CRDTDocument(site_id=tag)
    for op in ops:
        replica.apply_remote(op)
        if duplicate:
            replica.apply_remote(op)   # at-least-once delivery
    return replica.get_text()


all_ops = make_concurrent_ops(seed=11)
random.seed(7)

results = {}
for trial in range(300):
    order = all_ops[:]
    random.shuffle(order)             # commutativity + associativity in one shot:
    text = replay(order, f"r{trial}")  # an arbitrary interleaving of every op
    results.setdefault(text, 0)
    results[text] += 1

# ...and the same shuffles again, delivering every message twice
dup_results = set()
for trial in range(50):
    order = all_ops[:]
    random.shuffle(order)
    dup_results.add(replay(order, f"d{trial}", duplicate=True))

print(f"{len(all_ops)} operations from 3 sites (inserts AND deletes)")
print(f"  300 random delivery orders   → {len(results)} distinct document(s): {list(results)}")
print(f"   50 orders, every op twice   → {len(dup_results)} distinct document(s): {list(dup_results)}")

assert len(results) == 1, f"CRDT is not commutative/associative: {sorted(results)}"
assert dup_results == set(results), (
    f"CRDT is not idempotent: duplicate delivery gave {sorted(dup_results)} "
    f"instead of {sorted(results)}"
)
print("\n✅ commutative + associative (order never mattered) and idempotent (retries were free)")

# What idempotence is actually protecting you from: a merge that just splices
# the character in without checking whether it has seen it before.
class NaiveCRDTDocument(CRDTDocument):
    """apply_remote without the duplicate check — otherwise identical."""
    def apply_remote(self, op):
        if op["type"] != "insert":
            return super().apply_remote(op)
        c = CRDTChar(op["position_id"], op["char"], op["char_id"])
        i = 0
        while i < len(self.chars) and self.chars[i].sort_key() < c.sort_key():
            i += 1
        self.chars.insert(i, c)

naive = NaiveCRDTDocument(site_id="naive")
for op in all_ops:
    naive.apply_remote(op)
    naive.apply_remote(op)      # the network retried

print()
print(f"Same ops, same order, no duplicate check:")
print(f"  correct merge: '{next(iter(results))}'")
print(f"  naive merge:   '{naive.get_text()}'  ❌ every character delivered twice")

assert len(naive.get_text()) > len(next(iter(results))), (
    "the duplicate-delivery bug no longer reproduces — this demo is not showing anything"
)

## 📊 OT vs CRDTs: A Comparison

Let's compare the two approaches side by side:

In [ ]:
# Let's measure the memory difference
import sys
random.seed(42)

# Create a CRDT document, write 100 chars, then delete 50
crdt_doc = CRDTDocument(site_id="alice")
text = "The quick brown fox jumps over the lazy dog. " * 2  # ~90 chars
for i, ch in enumerate(text):
    crdt_doc.insert(i, ch)

visible_before = len(crdt_doc.get_text())
total_before = len(crdt_doc.get_all_chars())

# Delete every other character (simulate heavy editing)
delete_count = 0
for i in range(0, len(text), 2):
    idx = i - delete_count  # adjust for characters already removed from visible
    if idx < len(crdt_doc.get_text()):
        crdt_doc.delete(idx)
        delete_count += 1

visible_after = len(crdt_doc.get_text())
total_after = len(crdt_doc.get_all_chars())

print("📊 CRDT Memory Usage")
print("=" * 50)
print(f"Before deletions:")
print(f"  Visible chars: {visible_before}")
print(f"  Total in memory: {total_before}")
print()
print(f"After deleting {delete_count} characters:")
print(f"  Visible chars: {visible_after}")
print(f"  Total in memory: {total_after}  ← same! Tombstones still there")
print(f"  Memory overhead: {total_after - visible_after} tombstoned chars ({((total_after - visible_after) / total_after * 100):.0f}%)")
print()
print("💡 In OT, deleted text is just gone. In CRDTs, it stays as tombstones.")
print("   For long-lived documents with heavy editing, this adds up.")

assert total_after == total_before, "tombstones were reclaimed — then they are not tombstones"
assert visible_after == visible_before - delete_count, (
    f"expected {visible_before - delete_count} visible chars, got {visible_after}"
)
# Every character still has its own position id: 90 nested inserts is exactly
# where the float version collapsed at the top of this notebook.
ids = [c.position_id for c in crdt_doc.get_all_chars()]
assert len(set(ids)) == len(ids), (
    f"{len(ids) - len(set(ids))} characters share a position id — exact rationals "
    f"are supposed to make that impossible"
)

In [ ]:
# Summary comparison table

print("📊 OT vs CRDTs: When to Use Each")
print("=" * 70)
print()
print(f"{'Feature':<30} {'OT':<20} {'CRDTs':<20}")
print("-" * 70)

comparisons = [
    ("Central server",          "Required ⚠️",     "Not needed ✅"),
    ("Offline support",         "Limited ⚠️",      "Excellent ✅"),
    ("Memory usage",            "Low ✅",           "Higher (tombstones) ⚠️"),
    ("Implementation",          "Complex ⚠️",      "Complex ⚠️"),
    ("Peer-to-peer",            "No ❌",            "Yes ✅"),
    ("Scaling",                 "1 server/doc ⚠️", "Unlimited ✅"),
    ("Conflict quality",        "Good ✅",          "Can be awkward ⚠️"),
    ("Text editing fit",        "Excellent ✅",     "Good ✅"),
]

for feature, ot, crdt in comparisons:
    print(f"{feature:<30} {ot:<20} {crdt:<20}")

print()
print("Real-world choices:")
print("  • Google Docs → OT (central server, low memory, proven)")
print("  • Figma → CRDT-inspired (design ops, custom implementation)")
print("  • Apple Notes → CRDT (offline-first across iCloud devices)")
print("  • Yjs (open source) → CRDT (great for building your own)")

## 🧹 Cleanup

In [ ]:
print("🧹 No cleanup needed — all CRDT operations were in-memory.")

## 📚 Summary

### Key Takeaways

1. **CRDTs make operations commutative** — apply in any order, get the same result
2. **Unique position IDs** — each character gets an ID that never changes, even when neighbors are inserted
3. **Tombstones for deletion** — deleted characters stay in memory (trade-off: memory grows)
4. **No central server** — perfect for offline editing, peer-to-peer, and scaling
5. **Trade-off**: more memory, potentially awkward merge results at the same position

### What This Toy CRDT Does Not Do

Be honest about the gap between this notebook and Yjs or Automerge:

- **Tombstones are never collected.** Real CRDTs garbage-collect them once every
  replica is known to have seen the delete — which needs version vectors we don't have.
- **Position ids grow.** Our rationals gain digits with every nested insert.
  Logoot/LSEQ use variable-length id paths with the same problem, managed better.
- **Interleaving still happens** (see the `Hello,! world` merge above). RGA-style
  "insert after character X" references reduce it; nothing removes it entirely.
- **No causality tracking.** We patch over out-of-order deletes with a tombstone
  set; a real system carries version vectors and buffers ops until their
  dependencies arrive.
- **Per-character overhead is huge.** One Python object per character is fine for
  a lab and hopeless for a 50-page document; real implementations pack runs of
  characters into blocks.

### For System Design Interviews

- Default to **OT** for Google Docs-style problems (what they actually use)
- Mention **CRDTs** as an alternative if the interviewer asks about offline mode or P2P
- Know the trade-offs: OT = low memory + central server; CRDTs = more memory + no server

### Next Up

In **Notebook 3**, we'll build the **real-time collaboration experience** using WebSockets — connecting to our doc server, sending live edits, and seeing other users' cursors.